这里的代码blockA和blockB和blockC和blockD是分别用于计算Z/EDDY HEAT FLUX/EP FLUX/U并且把结果存入result里面供后面绘图调用。这里计算Z包括了B2000WCN气候态（仅仅1~12月平均）和BWCN的23年数据（逐月），其余的都不含气候态。 blockE是绘制位势高度叠加eddy stationary wave? blockF是绘制EP FLUX叠加EDDY HEAT FLUX负数等值线，注意这里的Fz要取负，因为计算出来的Fz大于0意味着随p减少增加（向上）。
需要非常注意的是这里的EPflux计算anomaly仅来自23年数据。我新开一个代码来计算整个206年气候态背景。
后面还添加了垂直残余速度和位势高度tendency的计算和绘制！！

In [ ]:
import os, glob
import numpy as np
import xarray as xr

# ============================================================
#  Process Z3:
#   (1) B2000WCN longrun hybrid Z3 -> monthly -> stdplev -> 12-month climatology
#   (2) BWCN hindcast/experiment h3 hybrid Z3 -> monthly -> stdplev time series
#  Output:
#   - Z3_B2000WCN002_clim_12mon_stdplev.nc
#   - Z3_BWCN002_monthly_stdplev.nc
# ============================================================

# ================= 配置区 =================
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

# Longrun climatology (yearly hybrid files you extracted)
B2000_Z3_HYB_DIR = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/Z3"
OUT_B2000_CLIM12 = os.path.join(OUT_DIR, "Z3_B2000WCN002_clim_12mon_stdplev.nc")

# Target experiment (BWCN h3 CAM files)
BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_BWCN_MONTHLY = os.path.join(OUT_DIR, "Z3_BWCN002_monthly_stdplev.nc")

# Standard pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# Reading strategy (netcdf4 + parallel=False to avoid "Not a valid ID")
ENGINE = "netcdf4"      # if unstable on your node, switch to "h5netcdf"
PARALLEL = False        # keep False
CHUNKS = {"time": 1}    # ok for monthly processing + dask ufunc
# =========================================


def safe_open_mfdataset(files):
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=PARALLEL,
        use_cftime=True,
        chunks=CHUNKS,
        engine=ENGINE,
    )
    return ds


def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]  # (lev)
    hybm = ds["hybm"]  # (lev)
    P0   = ds["P0"]    # scalar Pa
    PS   = ds["PS"]    # (time, lat, lon)
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid


def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    log(p) linear interpolation for each vertical column to target pressure levels.
    v_hyb, p_hyb: (..., lev), p in Pa
    returns: (..., plev)
    """
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending pressure
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},   # required for dask parallelized
    )

    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out


def monthly_mean_then_interp_to_stdplev(ds: xr.Dataset, varname="Z3") -> xr.DataArray:
    """
    ds must include: varname(time,lev,lat,lon), PS(time,lat,lon), hyam/hybm/P0
    output: monthly mean then log(p) interp to std plev -> (time, plev, lat, lon)
    """
    need = [varname, "PS", "hyam", "hybm", "P0"]
    missing = [v for v in need if v not in ds.variables]
    if missing:
        raise RuntimeError(f"Dataset missing vars: {missing}")

    # monthly mean aligned to month start
    ds_m = ds[need].resample(time="MS").mean()

    v_m = ds_m[varname]               # (time, lev, lat, lon)
    p_m = compute_pressure_mid(ds_m)  # (time, lev, lat, lon)

    v_std = interp_profile_logp(v_m, p_m, PLEV_STD_PA)
    v_std = v_std.transpose("time", "plev", "lat", "lon")

    v_std.name = f"{varname}_stdplev"
    v_std.attrs.update(
        units=v_m.attrs.get("units", ""),
        long_name=f"{varname}: monthly mean then log(p) interpolated to standard pressure levels",
    )
    return v_std


def write_nc(da: xr.DataArray, out_path: str, *, compress_level=4):
    """
    Write a DataArray to netcdf with compression.
    """
    name = da.name or "var"
    enc = {
        name: {"zlib": True, "complevel": int(compress_level), "dtype": "float32", "_FillValue": np.float32(np.nan)},
    }
    # keep coords dtype
    if "plev" in da.coords:
        enc["plev"] = {"dtype": "float64"}
    if "month_of_year" in da.coords:
        enc["month_of_year"] = {"dtype": "int32"}
    da.to_netcdf(out_path, encoding=enc)


# ============================================================
# (1) B2000WCN: yearly hybrid files -> monthly stdplev -> 12-month climatology
# ============================================================
b2000_files = sorted(glob.glob(os.path.join(B2000_Z3_HYB_DIR, "*.Z3.hybrid.nc")))
if len(b2000_files) == 0:
    raise RuntimeError(f"No B2000 yearly hybrid files found in: {B2000_Z3_HYB_DIR}")

ds_b2000 = safe_open_mfdataset(b2000_files)
z3_b2000_monthly_std = monthly_mean_then_interp_to_stdplev(ds_b2000, varname="Z3")  # (time, plev, lat, lon)

z3_b2000_clim12 = z3_b2000_monthly_std.groupby("time.month").mean("time")
z3_b2000_clim12 = z3_b2000_clim12.rename({"month": "month_of_year"})
z3_b2000_clim12["month_of_year"].attrs["long_name"] = "Month of year (1-12)"

write_nc(z3_b2000_clim12, OUT_B2000_CLIM12, compress_level=4)
ds_b2000.close()

print("Saved climatology (12 months):", OUT_B2000_CLIM12)


# ============================================================
# (2) BWCN: h3 files -> monthly stdplev time series
# ============================================================
bwcn_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(bwcn_files) == 0:
    raise RuntimeError(f"No BWCN h3 files found in: {BWCN_H3_DIR}")

ds_bwcn = safe_open_mfdataset(bwcn_files)
z3_bwcn_monthly_std = monthly_mean_then_interp_to_stdplev(ds_bwcn, varname="Z3")  # (time, plev, lat, lon)

write_nc(z3_bwcn_monthly_std, OUT_BWCN_MONTHLY, compress_level=4)
ds_bwcn.close()

print("Saved BWCN monthly time series:", OUT_BWCN_MONTHLY)


In [ ]:
import os, glob
import numpy as np
import xarray as xr
# 1. 引入进度条库
from dask.diagnostics import ProgressBar

# ============================================================
# BlockB — Eddy Heat Flux (v'T') from BWCN h3
# Output: (time, plev, lat) for ALL Latitudes (Global)
# ============================================================

# ---------------- paths ----------------
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"

# 修改文件名，体现这是全纬度数据
OUT_EHF_NC = os.path.join(OUT_DIR, "EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc")

# 标准 pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# dask chunks（time=1 比较稳）
CHUNKS = {"time": 1}

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]   # (lev)
    hybm = ds["hybm"]   # (lev)
    P0   = ds["P0"]     # scalar Pa
    PS   = ds["PS"]     # (time, lat, lon) Pa
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    v_hyb, p_hyb: (..., lev) 其中 p_hyb 单位 Pa
    p_tgt: (nplev,) 目标标准气压层 Pa
    return: (..., plev) log(p) 线性插值
    """
    p_tgt = np.asarray(p_tgt, float)

    def _interp_col(vcol, pcol):
        # 注意：这里接收到的是 numpy array
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        # 如果有效数据太少，直接返回全 NaN
        if m.sum() < 2:
            return np.full(p_tgt.shape, np.nan, float)

        p_use = pcol[m]
        v_use = vcol[m]

        # np.interp 要求自变量递增：按 pressure 升序
        idx = np.argsort(p_use)
        p_use = p_use[idx]
        v_use = v_use[idx]

        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        # ====================================================
        # 关键修复：必须指定新维度的长度，否则 apply_ufunc 会报错
        # ====================================================
        output_sizes={"plev": len(p_tgt)}, 
    )
    
    # 重新赋予坐标
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

# ---------------- main ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(h3_files) == 0:
    raise RuntimeError(f"No h3 files found in: {BWCN_H3_DIR}")

print(f"[BlockB_EHF] Found {len(h3_files)} h3 files.")
print("[BlockB_EHF] Opening multi-file dataset...")

# 只读需要的变量，降低 IO 压力
ds = xr.open_mfdataset(
    h3_files,
    combine="by_coords",
    parallel=True,
    use_cftime=True,
    chunks=CHUNKS,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

need_vars = ["V", "T", "PS", "P0", "hyam", "hybm"]
for v in need_vars:
    if v not in ds:
        raise RuntimeError(f"[BlockB_EHF] Missing variable '{v}' in BWCN h3 files.")

V = ds["V"]   # (time, lev, lat, lon)
T = ds["T"]   # (time, lev, lat, lon)

# 清理超大缺测值
V = V.where(np.abs(V) < 1e20)
T = T.where(np.abs(T) < 1e20)

# ---- zonal mean ----
V_zm = V.mean("lon")      # (time, lev, lat)
T_zm = T.mean("lon")      # (time, lev, lat)

# ---- eddy components ----
Vp = V - V_zm             # (time, lev, lat, lon)
Tp = T - T_zm             # (time, lev, lat, lon)

# ---- FULL v'T' then zonal mean (=> time, lev, lat) ----
# 注意：这就是 Eddy Heat Flux 的标准定义 [v*T*]
vt_zm = (Vp * Tp).mean("lon")   # (time, lev, lat)
vt_zm.name = "EHF_vTprime_zm"
vt_zm.attrs["long_name"] = "Eddy heat flux v'T' (zonal-mean of product)"
vt_zm.attrs["units"] = "K m s-1"  # CAM 里 V(m/s)*T(K)

# ---- pressure (zonal mean) for vertical interpolation ----
p_mid = compute_pressure_mid(ds)       # (time, lev, lat, lon)
p_zm  = p_mid.mean("lon")              # (time, lev, lat)

# ---- interpolate v'T' to standard pressure levels ----
print("[BlockB_EHF] Interpolating v'T' to standard pressure levels (log-p)...")
vt_std = interp_profile_logp(vt_zm, p_zm, PLEV_STD_PA)  # (time, lat, plev)

# 调整维度顺序符合习惯 (time, plev, lat)
vt_std = vt_std.transpose("time", "plev", "lat")

vt_std.name = "EHF_vTprime_stdplev"
vt_std.attrs.update(
    units="K m s-1",
    long_name="Eddy heat flux v'T' (zonal-mean) interpolated to standard pressure levels",
    interp_method="log(p) linear interpolation using zonal-mean mid-level pressure",
)

# ---- (已移除) select 40–80N ----
# 现在 vt_std 包含文件中所有的 latitude
print(f"[BlockB_EHF] Output shape: {vt_std.shape} (time, plev, lat)")

# ---- write to netcdf (compressed) ----
encoding = {
    "EHF_vTprime_stdplev": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
        "_FillValue": np.float32(np.nan),
    },
    "plev": {"dtype": "float64"},
}

print("[BlockB_EHF] Writing:", OUT_EHF_NC)
with ProgressBar():
    vt_std.to_netcdf(OUT_EHF_NC, encoding=encoding)

ds.close()
print("[BlockB_EHF] Done.")

In [ ]:
# =========================
# BlockC — EP Flux + EP flux divergence (Daily Calculation, Fixed Year Selection)
# =========================

import os, glob, sys, gc
import numpy as np
import xarray as xr
from tqdm import tqdm

# ---------------- paths ----------------
# 请确保路径与你实际环境一致
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_EP_NC   = os.path.join(OUT_DIR, "EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc")
AOSTOOLS_PY = "/mnt/data/aostools_functions.py" # 请确认此路径正确

# ---------------- settings ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

DO_UBAR = False   
WAVE    = -1      

# 导入自定义函数库
if os.path.exists(AOSTOOLS_PY):
    sys.path.insert(0, os.path.dirname(AOSTOOLS_PY))
from aostools_functions import ComputeEPfluxDiv

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """计算混合坐标系下的中间层气压"""
    hyam = ds["hyam"]
    hybm = ds["hybm"]
    P0   = ds["P0"]
    PS   = ds["PS"]
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    return p_mid

def interp_profile_logp_4d(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: np.ndarray) -> xr.DataArray:
    """在垂直方向上进行对数插值 (Hybrid -> Pressure Levels)"""
    p_tgt_pa = np.asarray(p_tgt_pa, float)
    
    def _interp_col(vcol, pcol):
        # 简单的列插值函数
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)
        # log-linear interpolation
        return np.interp(np.log(p_tgt_pa), np.log(p_use[idx]), v_use[idx], left=np.nan, right=np.nan)

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    out = out.assign_coords(plev=("plev", p_tgt_pa))
    return out

# ---------------- MAIN ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
print(f"[BlockC_EP] Found {len(h3_files)} files.")

# 1. Open Dataset (Lazy loading)
ds = xr.open_mfdataset(
    h3_files, combine="by_coords", parallel=True, use_cftime=True,
    chunks={"time": 30}, data_vars="minimal", coords="minimal"
)

# 2. Year Identification
years = np.unique(ds["time"].dt.year)
print(f"\n[DEBUG] Detected {len(years)} unique years: {years[0]} to {years[-1]}")

results_list = []

# 3. Processing Loop
for year in tqdm(years, desc="Processing Years", unit="year"):
    year_str = f"{year:04d}"
    
    # --- 3.1 Selection ---
    try:
        ds_yr_lazy = ds.sel(time=year_str)
    except Exception as e:
        tqdm.write(f"⚠️ Retrying selection with slice for year {year_str}...")
        try:
            ds_yr_lazy = ds.sel(time=slice(f"{year_str}-01-01", f"{year_str}-12-31"))
        except Exception as e2:
            tqdm.write(f"❌ Fatal error selecting year {year_str}: {e2}")
            continue

    # --- 3.2 Loading to Memory ---
    # 只加载需要的变量，减少内存压力
    try:
        ds_loaded = ds_yr_lazy[["U", "V", "T", "OMEGA", "PS", "P0", "hyam", "hybm"]].load()
    except Exception as e:
        tqdm.write(f"❌ Error loading data for {year_str}: {e}")
        continue
    
    # --- 【关键修复】检查时间步长 ---
    # numpy.gradient 需要至少3个点来计算二阶边界梯度
    if ds_loaded.sizes["time"] < 3:
        tqdm.write(f"⚠️ Warning: Year {year_str} has only {ds_loaded.sizes['time']} days (needs >= 3). Skipping to avoid gradient error.")
        continue

    # --- 3.3 Interpolation ---
    # 计算中间层气压
    p_mid = compute_pressure_mid(ds_loaded)
    
    # 插值到标准气压层
    u_std = interp_profile_logp_4d(ds_loaded["U"], p_mid, PLEV_STD_PA)
    v_std = interp_profile_logp_4d(ds_loaded["V"], p_mid, PLEV_STD_PA)
    t_std = interp_profile_logp_4d(ds_loaded["T"], p_mid, PLEV_STD_PA)
    w_std = interp_profile_logp_4d(ds_loaded["OMEGA"]/100.0, p_mid, PLEV_STD_PA) # Pa/s -> hPa/s 的转换在函数内部处理还是外部？
    # 注意：ComputeEPfluxDiv 内部通常期望 w 是 Pa/s 还是 hPa/s 取决于 aostools 实现。
    # 通常 aostools 文档写 w [hPa/s]。这里你除以 100 转换为 hPa/s 是对的。

    # 转为 numpy 数组 (Time, Plev, Lat, Lon)
    u_np = u_std.transpose("time", "plev", "lat", "lon").values
    v_np = v_std.transpose("time", "plev", "lat", "lon").values
    t_np = t_std.transpose("time", "plev", "lat", "lon").values
    w_np = w_std.transpose("time", "plev", "lat", "lon").values
    
    # --- 3.4 EP Flux Calculation ---
    # PLEV_STD_PA 是 Pa，但 ComputeEPfluxDiv 通常期望 hPa 作为 pres 输入用于计算
    # 代码中: pres=(PLEV_STD_PA / 100.0) -> 正确
    ep1, ep2, div1, div2 = ComputeEPfluxDiv(
        lat=ds_loaded["lat"].values,
        pres=(PLEV_STD_PA / 100.0), 
        u=u_np, v=v_np, t=t_np, w=w_np,
        do_ubar=DO_UBAR, wave=WAVE
    )

    # --- 3.5 Pack into Dataset ---
    ds_out_yr = xr.Dataset(
        data_vars=dict(
            EP1_cart   = (("time","plev","lat"), ep1, {"units":"m2 s-2", "long_name": "EP Flux y-component (Cartesian)"}),
            EP2_cart   = (("time","plev","lat"), ep2, {"units":"hPa m s-2", "long_name": "EP Flux z-component (Cartesian)"}),
            DIV1       = (("time","plev","lat"), div1, {"units":"m s-1 day-1", "long_name": "Div F (y-component contribution)"}),
            DIV2       = (("time","plev","lat"), div2, {"units":"m s-1 day-1", "long_name": "Div F (z-component contribution)"}),
            DIV_TOTAL  = (("time","plev","lat"), div1+div2, {"units":"m s-1 day-1", "long_name": "Total EP Flux Divergence"}),
        ),
        coords=dict(
            time = ds_loaded["time"],
            plev = ("plev", PLEV_STD_PA, {"units":"Pa"}),
            lat  = ds_loaded["lat"],
        )
    )
    
    results_list.append(ds_out_yr)
    
    # --- 3.6 Memory Cleanup ---
    del ds_loaded, u_std, v_std, t_std, w_std, u_np, v_np, t_np, w_np, ep1, ep2
    gc.collect()

# 4. Concatenate & Save
if len(results_list) > 0:
    print("\n[BlockC_EP] Concatenating all years...")
    ds_final = xr.concat(results_list, dim="time")
    
    ds_final.attrs = dict(source="BWCN.e122 CAM H3 Daily Data - EP Flux Analysis")
    
    # 设置压缩编码
    encoding = {vn: {"zlib": True, "complevel": 4, "_FillValue": np.nan, "dtype": "float32"} for vn in ds_final.data_vars}
    # 保持坐标轴精度
    encoding["plev"] = {"dtype": "float64"}
    if "time" in encoding: del encoding["time"] 
    
    print(f"[BlockC_EP] Saving to {OUT_EP_NC}...")
    ds_final.to_netcdf(OUT_EP_NC, encoding=encoding)
    print("[BlockC_EP] All Done!")
else:
    print("❌ Error: No results were generated.")

In [ ]:
import os, glob
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ============================================================
# BlockB2 — Zonal wind (U) zonal-mean -> std pressure levels
# Output: (time, plev, lat) for ALL Latitudes (Global)
# ============================================================

# ---------------- paths ----------------
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"

OUT_U_NC = os.path.join(OUT_DIR, "U_BWCN002_zm_ALL_LAT_stdplev_time_plev_lat.nc")

# 标准 pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# dask chunks（time=1 比较稳）
CHUNKS = {"time": 1}

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]   # (lev)
    hybm = ds["hybm"]   # (lev)
    P0   = ds["P0"]     # scalar Pa
    PS   = ds["PS"]     # (time, lat, lon) Pa
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    v_hyb, p_hyb: (..., lev) 其中 p_hyb 单位 Pa
    p_tgt: (nplev,) 目标标准气压层 Pa
    return: (..., plev) log(p) 线性插值
    """
    p_tgt = np.asarray(p_tgt, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt.shape, np.nan, float)

        p_use = pcol[m]
        v_use = vcol[m]

        # np.interp 要求自变量递增：按 pressure 升序
        idx = np.argsort(p_use)
        p_use = p_use[idx]
        v_use = v_use[idx]

        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": len(p_tgt)},
    )

    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

# ---------------- main ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(h3_files) == 0:
    raise RuntimeError(f"No h3 files found in: {BWCN_H3_DIR}")

print(f"[BlockB2_U] Found {len(h3_files)} h3 files.")
print("[BlockB2_U] Opening multi-file dataset...")

ds = xr.open_mfdataset(
    h3_files,
    combine="by_coords",
    parallel=True,
    use_cftime=True,
    chunks=CHUNKS,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

need_vars = ["U", "PS", "P0", "hyam", "hybm"]
for v in need_vars:
    if v not in ds:
        raise RuntimeError(f"[BlockB2_U] Missing variable '{v}' in BWCN h3 files.")

U = ds["U"]   # (time, lev, lat, lon) usually m/s in CAM

# 清理超大缺测值
U = U.where(np.abs(U) < 1e20)

# ---- zonal mean ----
U_zm = U.mean("lon")  # (time, lev, lat)
U_zm.name = "U_zm_hybrid"
U_zm.attrs["long_name"] = "Zonal mean zonal wind U"
# 尽量保留原 units；没有的话就补一个常见的
U_zm.attrs["units"] = U.attrs.get("units", "m s-1")

# ---- pressure (zonal mean) for vertical interpolation ----
p_mid = compute_pressure_mid(ds)   # (time, lev, lat, lon)
p_zm  = p_mid.mean("lon")          # (time, lev, lat)

# ---- interpolate U to standard pressure levels ----
print("[BlockB2_U] Interpolating U(zm) to standard pressure levels (log-p)...")
U_std = interp_profile_logp(U_zm, p_zm, PLEV_STD_PA)  # (time, lat, plev)

# 调整维度顺序 (time, plev, lat)
U_std = U_std.transpose("time", "plev", "lat")

U_std.name = "U_zm_stdplev"
U_std.attrs.update(
    units=U_zm.attrs.get("units", "m s-1"),
    long_name="Zonal mean zonal wind U interpolated to standard pressure levels",
    interp_method="log(p) linear interpolation using zonal-mean mid-level pressure",
)

print(f"[BlockB2_U] Output shape: {U_std.shape} (time, plev, lat)")

# ---- write to netcdf (compressed) ----
encoding = {
    "U_zm_stdplev": {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32",
        "_FillValue": np.float32(np.nan),
    },
    "plev": {"dtype": "float64"},
}

print(f"[BlockB2_U] Saving to {OUT_U_NC} ...")

# 真正触发计算并写入文件
# 使用 ProgressBar 可以看到进度条
with ProgressBar():
    U_std.to_netcdf(OUT_U_NC, format="NETCDF4", encoding=encoding)

print("[BlockB2_U] Done.")


In [ ]:
#计算w
import os, glob
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ============================================================
# BlockWSTAR — Residual vertical velocity omega* (TEM)
# Daily mean, interpolate to standard pressure levels
# Output: (time, plev, lat) for ALL latitudes
# ============================================================

OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
os.makedirs(OUT_DIR, exist_ok=True)

BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_WSTAR_NC = os.path.join(OUT_DIR, "OMEGASTAR_BWCN002_stdplev_daily_lat_ALL.nc")

ENGINE = "netcdf4"
CHUNKS = {"time": 1}   # 稳一点；如果太慢再改

PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

# ---- constants ----
a = 6.371e6
p0 = 100000.0
kappa = 0.2854

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]
    hybm = ds["hybm"]
    P0   = ds["P0"]
    PS   = ds["PS"]
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    """
    log(p) linear interpolation along lev -> plev
    v_hyb, p_hyb: (..., lev)  p in Pa
    """
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending p
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def daily_mean(ds, varlist):
    # 统一做 daily mean（如果原本就是 daily，这步不会伤）
    return ds[varlist].resample(time="1D").mean()

def theta_from_T(T, p_pa):
    # T(...,plev,lat,lon), p_pa(plev)
    # theta = T (p0/p)^kappa
    fac = (p0 / p_pa)**kappa
    return T * fac

def calc_omega_star_pressure_coords(omega, vtheta, theta, lat_deg, p_pa):
    """
    TEM residual vertical velocity in pressure coords:
    ω* = ω + (1/(a cosφ)) ∂/∂φ [ cosφ * (v'θ') / (∂θ/∂p) ]
    All inputs on (time, plev, lat).
    """
    lat = lat_deg
    lat_rad = np.deg2rad(lat)
    cosphi = np.cos(lat_rad)

    # dθ/dp along plev (Pa)
    # theta: (time, plev, lat)
    dtheta_dp = theta.differentiate("plev")  # K/Pa
    # guard
    dtheta_dp = xr.where(np.abs(dtheta_dp) < 1e-10, np.sign(dtheta_dp) * 1e-10, dtheta_dp)

    term = (vtheta * cosphi) / dtheta_dp  # (time, plev, lat)
    # derivative w.r.t. phi (radians)
    phi = xr.DataArray(lat_rad, dims=("lat",), coords={"lat": lat})
    dterm_dphi = term.differentiate("lat") / phi.differentiate("lat")  # approx d/dphi

    omega_star = omega + dterm_dphi / (a * cosphi)
    return omega_star

# ---------------- main ----------------
h3_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
if len(h3_files) == 0:
    raise RuntimeError(f"No BWCN h3 files found in: {BWCN_H3_DIR}")

print(f"[BlockWSTAR] Found {len(h3_files)} h3 files.")
print("[BlockWSTAR] Opening multi-file dataset...")

ds = xr.open_mfdataset(
    h3_files,
    combine="by_coords",
    parallel=False,          # 你之前遇到过 netcdf4 id 问题，这里保守
    use_cftime=True,
    chunks=CHUNKS,
    engine=ENGINE,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

need = ["V", "T", "OMEGA", "PS", "P0", "hyam", "hybm"]
for v in need:
    if v not in ds:
        raise RuntimeError(f"[BlockWSTAR] Missing variable '{v}' in dataset.")

# 1) daily mean
ds_d = daily_mean(ds, ["V", "T", "OMEGA", "PS", "P0", "hyam", "hybm"])
V = ds_d["V"].where(np.abs(ds_d["V"]) < 1e20)
T = ds_d["T"].where(np.abs(ds_d["T"]) < 1e20)
W = ds_d["OMEGA"].where(np.abs(ds_d["OMEGA"]) < 1e20)  # Pa/s

# 2) pressure mid on hybrid
p_mid = compute_pressure_mid(ds_d)  # (time, lev, lat, lon)

# 3) interpolate V, T, OMEGA to stdplev (still 3D/4D)
V_std = interp_profile_logp(V, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")
T_std = interp_profile_logp(T, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")
W_std = interp_profile_logp(W, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")

# 4) compute theta and eddy v'theta'
# zonal means
V_zm = V_std.mean("lon")
T_zm = T_std.mean("lon")
W_zm = W_std.mean("lon")  # Euler mean ω

theta = theta_from_T(T_std, T_std["plev"])          # (time, plev, lat, lon)
theta_zm = theta.mean("lon")

Vp = V_std - V_zm
thetap = theta - theta_zm
vtheta_zm = (Vp * thetap).mean("lon")              # (time, plev, lat)

# 5) omega*
omega_star = calc_omega_star_pressure_coords(
    omega=W_zm, vtheta=vtheta_zm, theta=theta_zm,
    lat_deg=W_zm["lat"], p_pa=W_zm["plev"]
)

omega_star = omega_star.transpose("time", "plev", "lat")
omega_star.name = "OMEGA_star_stdplev"
omega_star.attrs.update(
    units="Pa s-1",
    long_name="Residual vertical velocity omega* (TEM) in pressure coordinates",
    note="Computed from daily mean fields; v'theta' from eddy components on stdplev",
)

encoding = {
    "OMEGA_star_stdplev": {"zlib": True, "complevel": 4, "dtype": "float32", "_FillValue": np.float32(np.nan)},
    "plev": {"dtype": "float64"},
}

print("[BlockWSTAR] Writing:", OUT_WSTAR_NC)
with ProgressBar():
    omega_star.to_netcdf(OUT_WSTAR_NC, format="NETCDF4", encoding=encoding)

ds.close()
print("[BlockWSTAR] Done.")


In [ ]:
import glob
import numpy as np
import xarray as xr

# ---------------- paths & window ----------------
BWCN_H3_DIR = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
t0, t1 = "0007-12-01", "0008-04-01"
LAT0, LAT1 = 60.0, 90.0

# standard pressure (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000,
    15000, 20000, 30000, 50000, 70000, 100000
], dtype=float)

# dask chunks: daily window不长，但lon/lat/lev都很大，chunk稳一点
CHUNKS = {"time": 30}  # 你也可用 {"time": 10}

# ---------------- helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    """p_mid = hyam*P0 + hybm*PS (Pa)"""
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: np.ndarray) -> xr.DataArray:
    """
    log(p) linear interpolation along lev axis.
    v_hyb, p_hyb: (..., lev)
    return: (..., plev)
    """
    p_tgt_pa = np.asarray(p_tgt_pa, float)
    nplev = len(p_tgt_pa)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)  # ascending p
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt_pa), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",                 # ✅ 关键：允许 chunked arrays
        output_dtypes=[float],
        output_sizes={"plev": nplev},        # ✅ 关键：新维度长度
    )
    out = out.assign_coords(plev=("plev", p_tgt_pa))
    out["plev"].attrs["units"] = "Pa"
    return out

def sel_latband(da: xr.DataArray, lat0: float, lat1: float) -> xr.DataArray:
    lat = da["lat"]
    dec = float(lat.values[0]) > float(lat.values[-1])
    return da.sel(lat=slice(lat1, lat0) if dec else slice(lat0, lat1))

def coslat_mean(da: xr.DataArray) -> xr.DataArray:
    w = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(w).mean("lat", skipna=True)

# ---------------- open only what we need ----------------
files = sorted(glob.glob(f"{BWCN_H3_DIR}/*.cam.h3.*.nc"))
if len(files) == 0:
    raise RuntimeError(f"No h3 files found in {BWCN_H3_DIR}")

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

ds = xr.open_mfdataset(
    files,
    combine="by_coords",
    decode_times=time_coder,   # ✅ 替代 use_cftime=True (deprecated)
    chunks=CHUNKS,
    parallel=True,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

# subset time first to reduce work
ds = ds.sel(time=slice(t0, t1))

# daily mean (if original is sub-daily)
ds = ds.resample(time="1D").mean()

# check required vars
need = ["Z3", "PS", "P0", "hyam", "hybm"]
for v in need:
    if v not in ds:
        raise RuntimeError(f"Missing '{v}' in dataset.")

# ---------------- compute ----------------
p_mid = compute_pressure_mid(ds)          # (time, lev, lat, lon)
Z_hyb = ds["Z3"]                          # (time, lev, lat, lon)

# zonal mean first (reduces one dim BEFORE interpolation -> faster)
Z_zm = Z_hyb.mean("lon")                  # (time, lev, lat)
p_zm = p_mid.mean("lon")                  # (time, lev, lat)

# vertical interp to std plev
Z_std = interp_logp(Z_zm, p_zm, PLEV_STD_PA)   # (time, lat, plev)
Z_std = Z_std.transpose("time", "plev", "lat") # (time, plev, lat)

# polar cap mean 60–90N
Zcap = coslat_mean(sel_latband(Z_std, LAT0, LAT1))  # (time, plev)

# dZ/dt (m/day): use time in seconds
# 注意：cftime 不能直接 .astype('datetime64')，这里用数值差
t = Zcap["time"].values
tsec = np.array([(tt - t[0]).total_seconds() for tt in t], dtype=float)

dZdt = np.gradient(Zcap.values, tsec, axis=0) * 86400.0  # m/s -> m/day

dZdt = xr.DataArray(
    dZdt,
    dims=("time", "plev"),
    coords={"time": Zcap["time"], "plev": Zcap["plev"]},
    name="dZdt",
    attrs={"units": "m day-1", "long_name": "60–90N geopotential height tendency"}
)

# optional: close ds (dZdt is now realized lazily; if you want compute now: dZdt = dZdt.compute())
ds.close()

print(dZdt)


In [ ]:
# 300 hPa height anomaly + climatological stationary waves (polar stereo)
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point


# ================== 配置区 ==================
BWCN_MONTHLY_NC = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Z3_BWCN002_monthly_stdplev.nc"
B2000_CLIM12_NC = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Z3_B2000WCN002_clim_12mon_stdplev.nc"
OUT_FIG = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Fig7_abcde_Z300_anom_stationarywaves.png"

PLEV_TARGET = 30000.0

MONTHS = [
    ("Nov 0007", "0007-11-01"),
    ("Dec 0007", "0007-12-01"),
    ("Jan 0008", "0008-01-01"),
    ("Feb 0008", "0008-02-01"),
    ("Mar 0008", "0008-03-01"),
]

LAT_MIN = 20
LAT_MAX = 90

ANOM_VMAX = 200
ANOM_LEVELS = np.linspace(-ANOM_VMAX, ANOM_VMAX, 21)

WAVE_LEVELS = np.arange(-200, 201, 40)
# ===========================================


def month_from_timestr(timestr: str) -> int:
    return int(timestr.split("-")[1])


def pick_plev_2d_or_3d(da: xr.DataArray, plev_target: float) -> xr.DataArray:
    """
    Select nearest plev and DROP the plev dimension reliably.
    Return: if input has time dim, output keeps time; otherwise no time.
    """
    if "plev" not in da.dims:
        raise ValueError(f"No 'plev' in dims={da.dims}")

    out = da.sel(plev=plev_target, method="nearest").squeeze(drop=True)
    return out


def ensure_lat_ascending(da: xr.DataArray) -> xr.DataArray:
    # contourf 用 1D lat 时，lat 升序最省心
    if da["lat"].values[0] > da["lat"].values[-1]:
        da = da.sortby("lat")
    return da


def add_cyclic_2d(da2d: xr.DataArray):
    """
    da2d must be (lat, lon)
    return: data_cyc(2D np), lon_cyc(1D np)
    """
    da2d = da2d.squeeze(drop=True)
    if da2d.ndim != 2 or tuple(da2d.dims) != ("lat", "lon"):
        raise ValueError(f"add_cyclic_2d expects (lat, lon) 2D, got dims={da2d.dims}, shape={da2d.shape}")

    lon = da2d["lon"].values
    data_cyc, lon_cyc = add_cyclic_point(da2d.values, coord=lon)
    return data_cyc, lon_cyc


# ============ 读取数据 ============
ds_bw = xr.open_dataset(BWCN_MONTHLY_NC, use_cftime=True)
ds_cl = xr.open_dataset(B2000_CLIM12_NC, use_cftime=True)

Zname = "Z3_stdplev"
Z_bw = ds_bw[Zname]   # (time, plev, lat, lon)
Z_cl = ds_cl[Zname]   # (month_of_year, plev, lat, lon)

# 取 300 hPa（并确保 plev 维被 drop）
Z_bw_300 = pick_plev_2d_or_3d(Z_bw, PLEV_TARGET)   # (time, lat, lon)
Z_cl_300 = pick_plev_2d_or_3d(Z_cl, PLEV_TARGET)   # (month_of_year, lat, lon)

# 纬度裁剪（兼容 lat 升/降序）
Z_bw_300 = ensure_lat_ascending(Z_bw_300)
Z_cl_300 = ensure_lat_ascending(Z_cl_300)
Z_bw_300 = Z_bw_300.sel(lat=slice(LAT_MIN, LAT_MAX))
Z_cl_300 = Z_cl_300.sel(lat=slice(LAT_MIN, LAT_MAX))


# ============ 画图 ============
proj = ccrs.NorthPolarStereo()
pc = ccrs.PlateCarree()

fig, axes = plt.subplots(
    1, 5, figsize=(17, 4.2),
    subplot_kw=dict(projection=proj),
    constrained_layout=True
)

cf_last = None

for i, (title, tstr) in enumerate(MONTHS):
    ax = axes[i]

    # 选 BWCN 该月：结果应为 (lat, lon)；但我们再 squeeze 一次防呆
    Zm = Z_bw_300.sel(time=tstr).squeeze(drop=True)

    # 选 climatology 同月
    mo = month_from_timestr(tstr)
    Zc = Z_cl_300.sel(month_of_year=mo).squeeze(drop=True)

    # anomaly: (lat, lon)
    anom = (Zm - Zc).squeeze(drop=True)

    # stationary wave: climatological eddy height = Zc - zonalmean(Zc)
    wave = (Zc - Zc.mean("lon")).squeeze(drop=True)

    # cyclic
    anom_cyc, lon_cyc = add_cyclic_2d(anom)
    wave_cyc, _ = add_cyclic_2d(wave)

    lats = anom["lat"].values

    # 背景填色：anomaly
    cf = ax.contourf(
        lon_cyc, lats, anom_cyc,
        levels=ANOM_LEVELS,
        cmap="RdBu_r",
        extend="both",
        transform=pc
    )
    cf_last = cf

    # 叠加等值线：stationary waves
    ax.contour(
        lon_cyc, lats, wave_cyc,
        levels=WAVE_LEVELS,
        colors="k",
        linewidths=0.8,
        transform=pc
    )

    ax.set_title(f"({chr(97+i)}) {title}", fontsize=11)
    ax.set_extent([-180, 180, LAT_MIN, LAT_MAX], crs=pc)
    ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.6)
    ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=0.3)
    ax.gridlines(draw_labels=False, linewidth=0.3, alpha=0.5)

cbar = fig.colorbar(cf_last, ax=axes, orientation="vertical", shrink=0.88, pad=0.02)
cbar.set_label("Height Anomaly (m)")

fig.suptitle("300 hPa Height Anomalies (colors) & Climatological Stationary Waves (contours)", fontsize=13)

os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200)
plt.show()


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================================================
# Fig(f)-style plot: Fz standardized anomalies (blue-white-red, -4..4)
# + dashed contour: v'T' = 0 (boundary of negative heat flux)
# Winter: 0007/0008 (model calendar)
# ============================================================

# ---------------- paths ----------------
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
EP_NC   = f"{OUT_DIR}/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
EHF_NC  = f"{OUT_DIR}/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"

# ✅ output figure path
OUT_FIG = os.path.join(OUT_DIR, "Fig_f_Fz_std_anom_with_vT0.png")

# ---------------- settings ----------------
t0, t1 = "0007-11-01", "0008-04-30"     # change to "0008-03-31" if you want Nov–Mar only
LAT0, LAT1 = 40.0, 80.0
ZLIM = 4.0

# ---------------- helpers ----------------
def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- load ----------------
ds_ep  = xr.open_dataset(EP_NC)
ds_ehf = xr.open_dataset(EHF_NC)

Fp = ds_ep["EP2_cart"]                 # pressure-vertical EP flux component (p-positive downward)
vT = ds_ehf["EHF_vTprime_stdplev"]     # eddy heat flux v'T'

Fp, vT = xr.align(Fp, vT, join="inner")

# Flip sign so that "upward positive" for plotting
Fz_plot = -Fp

# ---------------- time window ----------------
Fz_win = Fz_plot.sel(time=slice(t0, t1))
vT_win = vT.sel(time=slice(t0, t1))

nt = Fz_win.sizes["time"]
if nt == 0:
    raise RuntimeError(f"Empty time slice {t0}..{t1}")

x = np.arange(nt)
time_str = np.array([str(tt)[:10] for tt in Fz_win["time"].values])

# ---------------- 40–80N mean ----------------
Fz_40_80 = coslat_weighted_mean(sel_latband(Fz_win, LAT0, LAT1))  # (time, plev)
vT_40_80 = coslat_weighted_mean(sel_latband(vT_win, LAT0, LAT1))  # (time, plev)


# ---------------- standardize Fz using Daily Climatology (DOY) ----------------
Fz_all_40_80 = coslat_weighted_mean(sel_latband(Fz_plot, LAT0, LAT1)) # (time, plev)

# 1. 计算 23 年里每一天的历日气候态 (1-365)
clim_mu = Fz_all_40_80.groupby("time.dayofyear").mean("time")
clim_sd = Fz_all_40_80.groupby("time.dayofyear").std("time")

# 2. 匹配目标窗口 (0007/0008) 每一天对应的气候态
sig_doy = Fz_40_80.time.dt.dayofyear
mu_matched = clim_mu.sel(dayofyear=sig_doy)
sd_matched = clim_sd.sel(dayofyear=sig_doy).where(lambda x: x > 1e-12)

# 3. 计算标准化距平 (利用 values 避开坐标冲突，然后重新封装)
Fz_stdzd_vals = (Fz_40_80.values - mu_matched.values) / sd_matched.values
Fz_stdzd = xr.DataArray(Fz_stdzd_vals, coords=Fz_40_80.coords, dims=Fz_40_80.dims)

# ---------------- arrays for plotting ----------------
plev = Fz_stdzd["plev"].values
Z_Fz = Fz_stdzd.transpose("plev", "time").values  # (plev, time)
Z_vT = vT_40_80.transpose("plev", "time").values  # (plev, time)

# ---------------- plot ----------------
fig, ax = plt.subplots(figsize=(14, 4.8))

levels = np.linspace(-ZLIM, ZLIM, 17)
cmap = plt.get_cmap("RdBu_r")  # blue-white-red
norm = TwoSlopeNorm(vmin=-ZLIM, vcenter=0.0, vmax=ZLIM)

cf = ax.contourf(x, plev, Z_Fz, levels=levels, cmap=cmap, norm=norm, extend="both")

# dashed: v'T' = 0 contour
ax.contour(x, plev, Z_vT, levels=[0.0], colors="k", linestyles="--", linewidths=1.3)

ax.set_yscale("log")
ax.invert_yaxis()

# y ticks labeled in hPa (coords are Pa)
yticks_pa = np.array([100000, 70000, 50000, 30000, 20000, 10000, 7000, 5000, 3000, 2000, 1000, 500, 300, 200, 100, 50, 10])
yticks_pa = yticks_pa[(yticks_pa >= float(plev.min())) & (yticks_pa <= float(plev.max()))]
ax.set_yticks(yticks_pa)
ax.set_yticklabels([f"{int(p/100)}" for p in yticks_pa])
ax.set_ylabel("Pressure (hPa)")

# x ticks with model-date strings (x itself is index)
tick_idx = np.linspace(0, nt-1, 8, dtype=int)
ax.set_xticks(tick_idx)
ax.set_xticklabels(time_str[tick_idx])
ax.set_xlabel("Model day (labels are model calendar dates)")

ax.set_title("40–80°N $F_z$ (upward-positive) standardized anomalies (Oct–Mar), winter 0007/0008\nDashed: $\\overline{v'T'}=0$")

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Std Anom")
cbar.set_ticks([-4, -2, 0, 2, 4])

plt.tight_layout()

# ✅ save then show
os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

ds_ep.close()
ds_ehf.close()


In [ ]:
import os, sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
U_NC    = f"{OUT_DIR}/U_BWCN002_zm_ALL_LAT_stdplev_time_plev_lat.nc"
EP_NC   = f"{OUT_DIR}/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"

# ✅ output figure path
OUT_FIG = os.path.join(OUT_DIR, "Fig_EPflux_arrows_on_Uzm_5mon.png")

AOSTOOLS_PY = "/mnt/data/aostools_functions.py"
if os.path.exists(AOSTOOLS_PY):
    sys.path.insert(0, os.path.dirname(AOSTOOLS_PY))
from aostools_functions import PlotEPfluxArrows

# ---- winter window ----
t0, t1 = "0007-11-01", "0008-03-31"
LAT_MIN, LAT_MAX = 0.0, 90.0
P_HPA_MIN, P_HPA_MAX = 0.1, 1000.0

U_VMIN, U_VMAX = -40.0, 80.0
LAT_STEP = 3
P_STEP   = 2

# ---- load ----
ds_u  = xr.open_dataset(U_NC)
ds_ep = xr.open_dataset(EP_NC)

Uzm = ds_u["U_zm_stdplev"]
EP1 = ds_ep["EP1_cart"]
EP2 = ds_ep["EP2_cart"]

Uzm, EP1, EP2 = xr.align(Uzm, EP1, EP2, join="inner")

# ---- subset ----
Uzm = Uzm.sel(time=slice(t0, t1)).sel(lat=slice(LAT_MIN, LAT_MAX))
EP1 = EP1.sel(time=slice(t0, t1)).sel(lat=slice(LAT_MIN, LAT_MAX))
EP2 = EP2.sel(time=slice(t0, t1)).sel(lat=slice(LAT_MIN, LAT_MAX))

plev_pa = Uzm["plev"]
mask_p = (plev_pa/100.0 >= P_HPA_MIN) & (plev_pa/100.0 <= P_HPA_MAX)
plev_keep = plev_pa.where(mask_p, drop=True)

Uzm = Uzm.sel(plev=plev_keep)
EP1 = EP1.sel(plev=plev_keep)
EP2 = EP2.sel(plev=plev_keep)

# ---- monthly means ----
U_m   = Uzm.resample(time="MS").mean().isel(time=slice(0, 5))
EP1_m = EP1.resample(time="MS").mean().isel(time=slice(0, 5))
EP2_m = EP2.resample(time="MS").mean().isel(time=slice(0, 5))

if U_m.sizes["time"] < 5:
    raise RuntimeError(f"Monthly series has only {U_m.sizes['time']} points; check t0/t1.")

# ---- titles ----
mon_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
titles = []
for tt in U_m["time"].values:
    s = str(tt)[:10]
    yy = int(s[:4]); mm = int(s[5:7])
    titles.append(f"{mon_names.get(mm, mm)} {yy:04d}")

# coords
lat_all   = U_m["lat"].values
p_hpa_all = (U_m["plev"].values / 100.0)
u_levels = np.linspace(U_VMIN, U_VMAX, 25)

# ---- plot ----
fig, axes = plt.subplots(1, 5, figsize=(17.5, 4.2), sharey=True)
plt.subplots_adjust(wspace=0.08)

mappable = None

for i, ax in enumerate(axes):
    U1 = U_m.isel(time=i).squeeze(drop=True)    # (plev, lat)
    e1 = EP1_m.isel(time=i).squeeze(drop=True)
    e2 = EP2_m.isel(time=i).squeeze(drop=True)

    # background wind
    U_plot = U1.transpose("plev", "lat").values
    cf = ax.contourf(
        lat_all, p_hpa_all, U_plot,
        levels=u_levels,
        cmap="RdBu_r",
        extend="both"
    )
    mappable = cf

    # subsample EP flux (2D: plev,lat)
    e1_s = e1.isel(lat=slice(None, None, LAT_STEP), plev=slice(None, None, P_STEP)).transpose("plev", "lat")
    e2_s = e2.isel(lat=slice(None, None, LAT_STEP), plev=slice(None, None, P_STEP)).transpose("plev", "lat")

    lat_s   = e1_s["lat"].values                      # (nlat,)
    p_s_hpa = (e1_s["plev"].values / 100.0)           # (nplev,)

    # PlotEPfluxArrows expects 2D x/y grids matching ep1/ep2 shape
    X2, Y2 = np.meshgrid(lat_s, p_s_hpa)              # (nplev, nlat)

    PlotEPfluxArrows(
        x=X2,
        y=Y2,
        ep1=e1_s.values,                               # (nplev, nlat)
        ep2=e2_s.values,                               # (nplev, nlat)
        fig=fig, ax=ax,
        xlim=(LAT_MIN, LAT_MAX),
        ylim=(P_HPA_MAX, P_HPA_MIN),
        xscale="linear",
        yscale="log",
        invert_y=True,
        scale=None,  # if arrows too long/short, try e.g. 5e7 (bigger -> shorter)
    )

    ax.set_title(titles[i], fontsize=11)
    ax.set_xlim(LAT_MIN, LAT_MAX)
    ax.set_xlabel("Latitude")

    # DO NOT invert y here; PlotEPfluxArrows already did it
    ax.set_yscale("log")
    ax.set_ylim(P_HPA_MAX, P_HPA_MIN)

    ax.grid(alpha=0.25)
    if i == 0:
        ax.set_ylabel("Pressure (hPa)")

# colorbar
cax = fig.add_axes([0.92, 0.15, 0.015, 0.72])
cb = fig.colorbar(mappable, cax=cax)
cb.set_label("Zonal Mean Zonal Wind (m/s)")

# ✅ save then show
os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

ds_u.close()
ds_ep.close()


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ===================== paths =====================
OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
U_NC    = f"{OUT_DIR}/U_BWCN002_zm_ALL_LAT_stdplev_time_plev_lat.nc"
EP_NC   = f"{OUT_DIR}/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"

# ✅ output figure path
OUT_FIG = os.path.join(OUT_DIR, "Fig_Uzm_EPFD_1hPa_10hPa_time_lat.png")

# ===================== settings =====================
t0, t1 = "0007-11-01", "0008-03-31"     # 0007/0008 winter (Nov–Mar)
LAT0, LAT1 = 20.0, 90.0                # paper shows ~20–90N; change if you want 0–90

# target levels (Pa)
PLEV_1HPA_PA  = 100.0
PLEV_10HPA_PA = 1000.0

# wind color limits (m/s) — match paper-ish
U_VMIN, U_VMAX = 0.0, 100.0

# EPFD contour levels (m/s/day)
lev_pos = np.array([8, 16, 32, 64], dtype=float)
lev_neg = -lev_pos[::-1]

# ===================== helpers =====================
def sel_latband(da, lat0, lat1):
    lat = da["lat"]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel(lat=slc)
    return out

def nearest_plev(da, plev_pa):
    return da.sel(plev=plev_pa, method="nearest")

# ===================== load =====================
ds_u  = xr.open_dataset(U_NC)
ds_ep = xr.open_dataset(EP_NC)

Uzm = ds_u["U_zm_stdplev"]       # (time, plev, lat)
DIV = ds_ep["DIV_TOTAL"]         # (time, plev, lat) m/s/day

Uzm, DIV = xr.align(Uzm, DIV, join="inner")

# time & lat window
Uzm = sel_latband(Uzm.sel(time=slice(t0, t1)), LAT0, LAT1)
DIV = sel_latband(DIV.sel(time=slice(t0, t1)), LAT0, LAT1)

# pick 1 hPa / 10 hPa
U_1  = nearest_plev(Uzm, PLEV_1HPA_PA)      # (time, lat)
U_10 = nearest_plev(Uzm, PLEV_10HPA_PA)     # (time, lat)

D_1  = nearest_plev(DIV, PLEV_1HPA_PA)      # (time, lat)
D_10 = nearest_plev(DIV, PLEV_10HPA_PA)     # (time, lat)

# x-axis: encode time as integer index (avoid cftime issues)
nt = U_1.sizes["time"]
x = np.arange(nt)
time_labels = np.array([str(tt)[:10] for tt in U_1["time"].values])

lat = U_1["lat"].values

# to arrays with shape (lat, time) for contourf
U1_plot  = U_1.transpose("lat", "time").values
U10_plot = U_10.transpose("lat", "time").values
D1_plot  = D_1.transpose("lat", "time").values
D10_plot = D_10.transpose("lat", "time").values

# ===================== plot =====================
fig, axes = plt.subplots(2, 1, figsize=(15, 3), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.08)

# ---- colormap ----
cmap = "Reds"

# ---- panel (f): 1 hPa ----
ax = axes[0]
cf = ax.contourf(
    x, lat, U1_plot,
    levels=np.linspace(U_VMIN, U_VMAX, 26),
    cmap=cmap,
    extend="both"
)

ax.contour(x, lat, D1_plot,  levels=lev_neg, colors="k", linestyles="--", linewidths=1.0)
ax.contour(x, lat, D1_plot,  levels=lev_pos, colors="k", linestyles="-",  linewidths=1.0)

ax.text(0.01, 0.08, "1 hPa", transform=ax.transAxes, fontsize=12, weight="bold")
ax.set_ylabel("Latitude")
ax.set_ylim(LAT0, LAT1)

# ---- panel (g): 10 hPa ----
ax = axes[1]
ax.contourf(
    x, lat, U10_plot,
    levels=np.linspace(U_VMIN, U_VMAX, 26),
    cmap=cmap,
    extend="both"
)

ax.contour(x, lat, D10_plot, levels=lev_neg, colors="k", linestyles="--", linewidths=1.0)
ax.contour(x, lat, D10_plot, levels=lev_pos, colors="k", linestyles="-",  linewidths=1.0)

ax.text(0.01, 0.08, "10 hPa", transform=ax.transAxes, fontsize=12, weight="bold")
ax.set_ylabel("Latitude")

# ---- x ticks as model dates ----
tick_idx = np.linspace(0, nt - 1, 6, dtype=int)
axes[1].set_xticks(tick_idx)
axes[1].set_xticklabels(time_labels[tick_idx])
axes[1].set_xlabel("Model time")

# ---- colorbar ----
cbar = fig.colorbar(cf, ax=axes, pad=0.02, aspect=30)
cbar.set_label("Zonal Mean Zonal Wind (m/s)")

# ✅ save then show
os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

ds_u.close()
ds_ep.close()


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

OUT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
WSTAR_NC = f"{OUT_DIR}/OMEGASTAR_BWCN002_stdplev_daily_lat_ALL.nc"
EHF_NC   = f"{OUT_DIR}/EHF_BWCN002_vTprime_ALL_LAT_stdplev_time_lat.nc"

OUT_FIG = os.path.join(OUT_DIR, "Fig9a_omega_star_0007_0008.png")

t0, t1 = "0007-12-01", "0008-04-01"
LAT0, LAT1 = 60.0, 90.0
P_MIN, P_MAX = 1.0, 100.0  # hPa
W_LIM = 2.0                # mPa/s

def sel_latband(da, lat0, lat1):
    lat = da["lat"]
    dec = lat[0] > lat[-1]
    return da.sel(lat=slice(lat1, lat0) if dec else slice(lat0, lat1))

def coslat_mean(da):
    w = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(w).mean("lat")

# ---- load ----
wds = xr.open_dataset(WSTAR_NC)
eds = xr.open_dataset(EHF_NC)

w = wds["OMEGA_star_stdplev"].sel(time=slice(t0, t1))
vT = eds["EHF_vTprime_stdplev"].sel(time=slice(t0, t1))

# pressure range
mask_p = (w["plev"]/100 >= P_MIN) & (w["plev"]/100 <= P_MAX)
w = w.sel(plev=w["plev"].where(mask_p, drop=True))
vT = vT.sel(plev=w["plev"])

# polar cap mean
wcap = coslat_mean(sel_latband(w, LAT0, LAT1)) * 1000.0  # Pa/s → mPa/s

# dashed line: v'T' at 60N
vT_60 = vT.sel(lat=60, method="nearest")

# ---- plotting ----
x = np.arange(wcap.sizes["time"])
time_labels = [str(t)[:10] for t in wcap["time"].values]
plev_hpa = wcap["plev"].values / 100.0

Z = wcap.transpose("plev", "time").values
C = vT_60.transpose("plev", "time").values

fig, ax = plt.subplots(figsize=(14, 3.5))

levels = np.linspace(-W_LIM, W_LIM, 17)
norm = TwoSlopeNorm(vmin=-W_LIM, vcenter=0, vmax=W_LIM)

cf = ax.contourf(x, plev_hpa, Z, levels=levels,
                 cmap="RdBu_r", norm=norm, extend="both")

ax.contour(x, plev_hpa, C, levels=[0], colors="k",
           linestyles="--", linewidths=1.3)

ax.axhline(50, color="k", linewidth=1)

ax.set_yscale("log")
ax.invert_yaxis()
ax.set_ylabel("Pressure [hPa]")
ax.set_title("60–90°N Residual Vertical Velocity")

ticks = np.linspace(0, len(x)-1, 6, dtype=int)
ax.set_xticks(ticks)
ax.set_xticklabels(np.array(time_labels)[ticks])

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label(r"$\omega^*$ [mPa s$^{-1}$]")

plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

wds.close()
eds.close()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np

OUT_FIG = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/Fig9b_Ztendency_0007_0008.png"

plev_hpa = dZdt["plev"].values / 100.0
Z = dZdt.transpose("plev", "time").values

x = np.arange(Z.shape[1])
time_labels = [str(t)[:10] for t in dZdt.time.values]

fig, ax = plt.subplots(figsize=(14, 3.5))

levels = np.linspace(-250, 250, 17)
norm = TwoSlopeNorm(vmin=-250, vcenter=0, vmax=250)

cf = ax.contourf(x, plev_hpa, Z,
                 levels=levels, cmap="RdBu_r",
                 norm=norm, extend="both")

ax.set_yscale("log")
ax.invert_yaxis()
ax.set_ylabel("Pressure [hPa]")
ax.set_title("Tendency of 60–90°N Geopotential Height")

ticks = np.linspace(0, len(x)-1, 6, dtype=int)
ax.set_xticks(ticks)
ax.set_xticklabels(np.array(time_labels)[ticks])

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Height tendency [m day$^{-1}$]")

plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ================= 配置区 =================
# 1. 你的 BWCN 实验组 NAM 与 AO 数据路径
BWCN_NAM_DIR = "/mnt/soclim0/public_data/weiji/BWCN/NAM"
NAM_NC = os.path.join(BWCN_NAM_DIR, "BWCN_Vertical_NAM.nc")
AO_CSV = os.path.join(BWCN_NAM_DIR, "BWCN_Daily_AO_Index.csv")

# 2. 之前计算的 EP Flux 结果 (保持不变)
EP_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
EP_NC = os.path.join(EP_DIR, "EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc")

# 3. 图片输出路径
OUT_FIG_3CD = os.path.join(EP_DIR, "Fig3_cd_square_BWCN_AO.png")
OUT_FIG_4   = os.path.join(EP_DIR, "Fig4_square_BWCN_AO.png")

LAT_FZ = (40.0, 80.0)  # Fz 的面积平均范围
# =========================================

# ----------------- 辅助函数 -----------------
def sel_lat_safe(da, lat_range):
    lat0, lat1 = lat_range
    if da["lat"].values[0] > da["lat"].values[-1]:
        return da.sel(lat=slice(lat1, lat0))
    else:
        return da.sel(lat=slice(lat0, lat1))

def get_coslat_weight(da, lat_range):
    da_sub = sel_lat_safe(da, lat_range)
    weights = np.cos(np.deg2rad(da_sub["lat"]))
    return da_sub.weighted(weights).mean("lat", skipna=True)

def standardize(da):
    return (da - da.mean()) / da.std()

def plot_scatter_generic(ax, x, y, labels, xlabel, ylabel, title, limit=(-2.5, 2.5), r_pos=(0.05, 0.95)):
    """通用散点图绘制函数，强制正方形比例"""
    ax.scatter(x, y, color="grey", alpha=0.5, s=20)
    
    for i, txt in enumerate(labels):
        ax.annotate(txt, (x[i], y[i]), fontsize=9, alpha=0.8, ha='center', va='bottom')
    
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.axvline(0, color='gray', linewidth=0.8)
    
    r, p = pearsonr(x, y)
    star = "*" if p < 0.05 else ""
    ax.text(r_pos[0], r_pos[1], f"r = {r:.3f}{star}", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', va='top', ha='left')
    
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(limit[0]-0.2, limit[1]+0.2, 100)
    ax.plot(x_line, m*x_line + b, color="lightgray", linestyle="--")
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc='left', fontweight='bold')
    
    ax.set_xlim(limit[0], limit[1])
    ax.set_ylim(limit[0], limit[1])
    ax.set_xticks([-2, -1, 0, 1, 2])
    ax.set_yticks([-2, -1, 0, 1, 2])
    
    # 强制图表内部网格为绝对正方形
    ax.set_box_aspect(1)
    ax.grid(True, linestyle=":", alpha=0.6)

# ================= 1. 数据读取与处理 =================
print("📥 读取 BWCN 实验组 NAM、AO 及 EP Flux 数据...")
ds_nam = xr.open_dataset(NAM_NC, use_cftime=True)
ds_ep  = xr.open_dataset(EP_NC, use_cftime=True)

# ----------------- 1.1 提取 JFM 50 hPa NAM (图3X轴) -----------------
nam_50 = ds_nam["NAM_Vertical"].sel(lev=50.0, method="nearest")
nam_50_jfm = nam_50.sel(time=nam_50.time.dt.month.isin([1, 2, 3]))
nam_50_jfm_yr = nam_50_jfm.groupby("time.year").mean("time")

nam_index = standardize(nam_50_jfm_yr)
nam_index.name = "NAM_50hPa"

# ----------------- 1.2 读取并计算 JFM AO 指数 (图3d Y轴) -----------------
df_ao = pd.read_csv(AO_CSV)
# 利用字符串切片提取年份和月份，完美避开 cftime 导致 pandas 报错的问题
df_ao['year'] = df_ao['Date'].str.slice(0, 4).astype(int)
df_ao['month'] = df_ao['Date'].str.slice(5, 7).astype(int)

# 筛选 JFM
df_ao_jfm = df_ao[df_ao['month'].isin([1, 2, 3])]
ao_jfm_yr = df_ao_jfm.groupby('year')['AO_Index'].mean()

# 转成 DataArray 方便对齐
ao_index = xr.DataArray(ao_jfm_yr.values, coords=[ao_jfm_yr.index], dims=["year"])
ao_index = standardize(ao_index)
ao_index.name = "AO_Index"

# ----------------- 1.3 提取 DJF 100 hPa Fz (图3c Y轴, 图4 Y轴) -----------------
Fz_100 = -1.0 * ds_ep["EP2_cart"].sel(plev=10000.0, method="nearest") 
Fz_100_midlat = get_coslat_weight(Fz_100, LAT_FZ) 

Fz_qs_100 = Fz_100_midlat.resample(time="QS-DEC").mean()
Fz_djf_100 = Fz_qs_100.sel(time=Fz_qs_100.time.dt.month == 12)

# 跨年归属：12月的 DJF 属于次年
djf_years_100 = Fz_djf_100["time"].dt.year.values + 1
Fz_djf_100 = Fz_djf_100.assign_coords(year=("time", djf_years_100)).swap_dims({"time": "year"})

fz100_index = standardize(Fz_djf_100)
fz100_index.name = "Fz_100hPa"

# ----------------- 1.4 提取 DJF 300 hPa Fz (图4 X轴) -----------------
Fz_300 = -1.0 * ds_ep["EP2_cart"].sel(plev=30000.0, method="nearest") 
Fz_300_midlat = get_coslat_weight(Fz_300, LAT_FZ) 

Fz_qs_300 = Fz_300_midlat.resample(time="QS-DEC").mean()
Fz_djf_300 = Fz_qs_300.sel(time=Fz_qs_300.time.dt.month == 12)

djf_years_300 = Fz_djf_300["time"].dt.year.values + 1
Fz_djf_300 = Fz_djf_300.assign_coords(year=("time", djf_years_300)).swap_dims({"time": "year"})

fz300_index = standardize(Fz_djf_300)
fz300_index.name = "Fz_300hPa"

# ================= 2. 数据对齐 =================
# 对齐 Figure 3cd 的年份 (NAM, AO, Fz100)
common_years_fig3 = np.intersect1d(nam_index.year.values, fz100_index.year.values)
common_years_fig3 = np.intersect1d(common_years_fig3, ao_index.year.values)

# 对齐 Figure 4 的年份 (Fz300, Fz100)
common_years_fig4 = np.intersect1d(fz300_index.year.values, fz100_index.year.values)

if len(common_years_fig3) == 0:
    raise ValueError("未找到匹配的共有年份进行作图！")

nam_plot_3 = nam_index.sel(year=common_years_fig3).values
fz100_plot_3 = fz100_index.sel(year=common_years_fig3).values
ao_plot_3 = ao_index.sel(year=common_years_fig3).values
year_labels_fig3 = [f"{y:02d}"[-2:] for y in common_years_fig3]

fz300_plot_4 = fz300_index.sel(year=common_years_fig4).values
fz100_plot_4 = fz100_index.sel(year=common_years_fig4).values
year_labels_fig4 = [f"{y:02d}"[-2:] for y in common_years_fig4]

# ================= 3. 开始绘图 =================

# ----------------- 3.1 绘制 Figure 3cd (完全正方形双列) -----------------
print(f"🎨 开始绘制图 3cd 到 {OUT_FIG_3CD}...")
fig3, axes3 = plt.subplots(1, 2, figsize=(11, 5.5)) 
plt.subplots_adjust(wspace=0.35)

plot_scatter_generic(axes3[0], nam_plot_3, fz100_plot_3, year_labels_fig3,
                     xlabel="JFM, 50 hPa NAM (σ)", ylabel="DJF, 100 hPa 40–80°N $F_z$ (σ)", title="(c)")

# 注意这里 Y 轴改为了 AO Index
plot_scatter_generic(axes3[1], nam_plot_3, ao_plot_3, year_labels_fig3,
                     xlabel="JFM, 50 hPa NAM (σ)", ylabel="JFM, AO Index (σ)", title="(d)")

plt.savefig(OUT_FIG_3CD, dpi=300, bbox_inches="tight")
plt.show(fig3)

# ----------------- 3.2 绘制 Figure 4 (正方形独立版) -----------------
print(f"🎨 开始绘制图 4 到 {OUT_FIG_4}...")
fig4, ax4 = plt.subplots(figsize=(6, 6))

plot_scatter_generic(ax4, fz300_plot_4, fz100_plot_4, year_labels_fig4,
                     xlabel="DJF, 300 hPa 40–80°N $F_z$ (σ)", ylabel="DJF, 100 hPa 40–80°N $F_z$ (σ)", title="")

plt.savefig(OUT_FIG_4, dpi=300, bbox_inches="tight")
plt.show(fig4)

print("✅ 修改完成！已成功接入 BWCN 实验组的新 NAM 与 AO 数据。")